# 🚀 Building Production APIs with FastAPI

**Create production-ready LLM APIs with FastAPI**

Learn to build REST APIs that expose your LLM applications to the world - the foundation of all production AI systems.

---

## 📋 Overview

**What you'll learn:**
- Build REST APIs with FastAPI
- Create LLM-powered endpoints
- Handle async operations
- Add validation and error handling
- Test and document your API

**Prerequisites:** 
- Completed LLM basics
- Understanding of HTTP/REST
- Python async/await basics

**Time estimate:** ⏱️ 60-90 minutes

**Difficulty:** 🟡 Intermediate

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. ✅ Build a FastAPI application from scratch
2. ✅ Create endpoints for LLM operations
3. ✅ Implement proper data validation
4. ✅ Add error handling and logging
5. ✅ Test and document your API
6. ✅ Understand production considerations

---

## 📖 Why FastAPI?

### FastAPI vs Other Frameworks

| Feature | FastAPI | Flask | Django |
|---------|---------|-------|--------|
| **Speed** | ⚡ Fastest | Medium | Slower |
| **Async** | ✅ Native | ❌ Limited | ✅ Yes |
| **Auto docs** | ✅ Built-in | ❌ Manual | ❌ Manual |
| **Validation** | ✅ Pydantic | ❌ Manual | ✅ Forms |
| **Type hints** | ✅ Required | ❌ Optional | ❌ Optional |

### Why FastAPI for LLMs?

1. **Async Support** - Perfect for slow LLM calls
2. **Automatic Validation** - Using Pydantic models
3. **Auto Documentation** - Swagger/OpenAPI built-in
4. **High Performance** - Fast response handling
5. **Type Safety** - Catch errors early

### Real-World Use Cases

- **Chatbot APIs** - Serve conversational AI
- **Text Processing** - Summarization, translation
- **RAG Systems** - Q&A over documents
- **Content Generation** - Blog posts, emails
- **Internal Tools** - Team productivity apps

---

## 🏗️ Step 1: Hello World API

Let's start with the simplest possible FastAPI application.

In [ ]:
# Create a simple API file
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel

# Initialize FastAPI app
app = FastAPI(
    title="My First LLM API",
    description="A simple API for LLM operations",
    version="1.0.0"
)

# Root endpoint
@app.get("/")
async def root():
    return {
        "message": "Welcome to the LLM API",
        "version": "1.0.0",
        "endpoints": ["/", "/health", "/docs"]
    }

# Health check endpoint
@app.get("/health")
async def health():
    return {"status": "healthy"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Save to file
with open('simple_api.py', 'w') as f:
    f.write(api_code)

print("✅ Created simple_api.py")
print("\nTo run:")
print("  python simple_api.py")
print("  # Or: uvicorn simple_api:app --reload")
print("\nThen visit:")
print("  http://localhost:8000      - Root endpoint")
print("  http://localhost:8000/docs - Interactive docs")

### 🔍 Understanding the Code

```python
app = FastAPI()           # Create FastAPI instance

@app.get("/")             # Define GET endpoint at root
async def root():          # Async function (can await)
    return {...}          # Return JSON automatically
```

**Key concepts:**
- `@app.get()` - HTTP GET request
- `async def` - Asynchronous function
- Return dict = JSON response
- Auto documentation at `/docs`

---

## 🤖 Step 2: LLM Completion Endpoint

Let's create an endpoint that calls an LLM!

In [ ]:
llm_api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

app = FastAPI(title="LLM API", version="1.0.0")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Request model (input validation)
class CompletionRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=2000, description="User prompt")
    max_tokens: Optional[int] = Field(default=100, ge=1, le=1000, description="Max tokens")
    temperature: Optional[float] = Field(default=0.7, ge=0.0, le=2.0, description="Temperature")
    
    class Config:
        schema_extra = {
            "example": {
                "prompt": "Explain quantum computing in one sentence",
                "max_tokens": 100,
                "temperature": 0.7
            }
        }

# Response model (output structure)
class CompletionResponse(BaseModel):
    success: bool
    response: str
    tokens_used: int
    model: str

@app.post("/api/v1/completion", response_model=CompletionResponse)
async def create_completion(request: CompletionRequest):
    """
    Generate text completion using LLM.
    
    - **prompt**: Your text prompt
    - **max_tokens**: Maximum tokens in response
    - **temperature**: Creativity level (0-2)
    """
    try:
        response = client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": request.prompt}],
            max_tokens=request.max_tokens,
            temperature=request.temperature
        )
        
        return CompletionResponse(
            success=True,
            response=response.choices[0].message.content,
            tokens_used=response.usage.total_tokens,
            model="mixtral-8x7b-32768"
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('llm_api.py', 'w') as f:
    f.write(llm_api_code)

print("✅ Created llm_api.py with LLM endpoint")
print("\nFeatures:")
print("  • Pydantic validation (automatic)")
print("  • Type hints (auto docs)")
print("  • Error handling")
print("  • Example requests in docs")

### 🧪 Testing the API

**Using curl:**
```bash
curl -X POST "http://localhost:8000/api/v1/completion" \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Hello world", "max_tokens": 50}'
```

**Using Python requests:**
```python
import requests

response = requests.post(
    "http://localhost:8000/api/v1/completion",
    json={
        "prompt": "Explain AI",
        "max_tokens": 100
    }
)
print(response.json())
```

**Using the interactive docs:**
1. Go to `http://localhost:8000/docs`
2. Click on the endpoint
3. Click "Try it out"
4. Fill in the request
5. Click "Execute"

---

## 🎯 Step 3: Complete Production API

Let's build a comprehensive API with multiple endpoints, error handling, and logging.

In [ ]:
production_api_code = '''
"""Production-ready LLM API with FastAPI"""

from fastapi import FastAPI, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, validator
from typing import Optional, List
from enum import Enum
import os
import time
import logging
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Initialize FastAPI
app = FastAPI(
    title="Production LLM API",
    description="A production-ready API for LLM operations",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Configure appropriately for production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize LLM client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Enums for validation
class TaskType(str, Enum):
    completion = "completion"
    summarization = "summarization"
    classification = "classification"

# Request models
class CompletionRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=2000)
    max_tokens: int = Field(default=100, ge=1, le=1000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    
    @validator("prompt")
    def prompt_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("Prompt cannot be empty")
        return v

class SummarizationRequest(BaseModel):
    text: str = Field(..., min_length=10, max_length=10000)
    max_length: int = Field(default=3, ge=1, le=10, description="Number of sentences")

class ClassificationRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=1000)
    categories: List[str] = Field(..., min_items=2, max_items=10)

# Response models
class CompletionResponse(BaseModel):
    success: bool
    response: str
    tokens_used: int
    model: str
    latency_ms: float

class SummarizationResponse(BaseModel):
    success: bool
    summary: str
    original_length: int
    summary_length: int
    tokens_used: int

class ClassificationResponse(BaseModel):
    success: bool
    category: str
    confidence: str
    tokens_used: int

# Health check
@app.get("/health")
async def health_check():
    """Check if the API is running."""
    return {
        "status": "healthy",
        "timestamp": time.time(),
        "version": "1.0.0"
    }

# Completion endpoint
@app.post("/api/v1/completion", response_model=CompletionResponse)
async def create_completion(request: CompletionRequest):
    """Generate text completion."""
    start_time = time.time()
    
    try:
        logger.info(f"Completion request: {request.prompt[:50]}...")
        
        response = client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": request.prompt}],
            max_tokens=request.max_tokens,
            temperature=request.temperature
        )
        
        latency = (time.time() - start_time) * 1000
        
        logger.info(f"Completion success: {response.usage.total_tokens} tokens, {latency:.0f}ms")
        
        return CompletionResponse(
            success=True,
            response=response.choices[0].message.content,
            tokens_used=response.usage.total_tokens,
            model="mixtral-8x7b-32768",
            latency_ms=latency
        )
    
    except Exception as e:
        logger.error(f"Completion error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

# Summarization endpoint
@app.post("/api/v1/summarize", response_model=SummarizationResponse)
async def summarize_text(request: SummarizationRequest):
    """Summarize text to specified length."""
    try:
        logger.info(f"Summarization request: {len(request.text)} chars")
        
        prompt = f"""Summarize the following text in exactly {request.max_length} sentences.
        
Text: {request.text}

Summary:"""
        
        response = client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.3
        )
        
        summary = response.choices[0].message.content
        
        return SummarizationResponse(
            success=True,
            summary=summary,
            original_length=len(request.text),
            summary_length=len(summary),
            tokens_used=response.usage.total_tokens
        )
    
    except Exception as e:
        logger.error(f"Summarization error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

# Classification endpoint
@app.post("/api/v1/classify", response_model=ClassificationResponse)
async def classify_text(request: ClassificationRequest):
    """Classify text into one of the provided categories."""
    try:
        logger.info(f"Classification request: {request.categories}")
        
        categories_str = ", ".join(request.categories)
        prompt = f"""Classify the following text into exactly ONE of these categories: {categories_str}
        
Text: {request.text}

Respond with ONLY the category name (one word).
Category:"""
        
        response = client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20,
            temperature=0.0
        )
        
        category = response.choices[0].message.content.strip()
        
        return ClassificationResponse(
            success=True,
            category=category,
            confidence="high",
            tokens_used=response.usage.total_tokens
        )
    
    except Exception as e:
        logger.error(f"Classification error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''

with open('production_api.py', 'w') as f:
    f.write(production_api_code)

print("✅ Created production_api.py")
print("\nFeatures:")
print("  ✅ Multiple endpoints (completion, summarize, classify)")
print("  ✅ Request validation with Pydantic")
print("  ✅ Error handling and logging")
print("  ✅ CORS middleware")
print("  ✅ Performance metrics (latency)")
print("  ✅ Auto-generated documentation")
print("\nRun with:")
print("  python production_api.py")
print("\nTest at:")
print("  http://localhost:8000/docs")

---

## 🧪 Testing Your API

Let's create a test client to verify our API works!

In [ ]:
test_client_code = '''
"""Test client for the LLM API"""

import requests
import json

BASE_URL = "http://localhost:8000"

def test_health():
    """Test health endpoint."""
    print("Testing /health...")
    response = requests.get(f"{BASE_URL}/health")
    print(f"Status: {response.status_code}")
    print(f"Response: {response.json()}\n")

def test_completion():
    """Test completion endpoint."""
    print("Testing /api/v1/completion...")
    
    payload = {
        "prompt": "Explain what FastAPI is in one sentence",
        "max_tokens": 50,
        "temperature": 0.7
    }
    
    response = requests.post(
        f"{BASE_URL}/api/v1/completion",
        json=payload
    )
    
    print(f"Status: {response.status_code}")
    result = response.json()
    print(f"Response: {result['response']}")
    print(f"Tokens: {result['tokens_used']}")
    print(f"Latency: {result['latency_ms']:.0f}ms\n")

def test_summarization():
    """Test summarization endpoint."""
    print("Testing /api/v1/summarize...")
    
    text = """Artificial intelligence has made remarkable progress in recent years. 
    Machine learning models can now perform tasks that were once thought impossible, 
    from generating human-like text to recognizing images with superhuman accuracy. 
    However, these advances also raise important questions about ethics, bias, 
    and the future of work."""
    
    payload = {
        "text": text,
        "max_length": 2
    }
    
    response = requests.post(
        f"{BASE_URL}/api/v1/summarize",
        json=payload
    )
    
    print(f"Status: {response.status_code}")
    result = response.json()
    print(f"Summary: {result['summary']}")
    print(f"Compression: {result['original_length']} → {result['summary_length']} chars\n")

def test_classification():
    """Test classification endpoint."""
    print("Testing /api/v1/classify...")
    
    payload = {
        "text": "I love this product! It works perfectly and exceeded my expectations.",
        "categories": ["positive", "negative", "neutral"]
    }
    
    response = requests.post(
        f"{BASE_URL}/api/v1/classify",
        json=payload
    )
    
    print(f"Status: {response.status_code}")
    result = response.json()
    print(f"Category: {result['category']}")
    print(f"Confidence: {result['confidence']}\n")

if __name__ == "__main__":
    print("=" * 80)
    print("API Test Suite")
    print("=" * 80)
    print()
    
    try:
        test_health()
        test_completion()
        test_summarization()
        test_classification()
        
        print("=" * 80)
        print("✅ All tests passed!")
        print("=" * 80)
    
    except requests.exceptions.ConnectionError:
        print("❌ Error: Could not connect to API")
        print("Make sure the API is running: python production_api.py")
    except Exception as e:
        print(f"❌ Error: {e}")
'''

with open('test_api.py', 'w') as f:
    f.write(test_client_code)

print("✅ Created test_api.py")
print("\nTo test your API:")
print("  1. Terminal 1: python production_api.py")
print("  2. Terminal 2: python test_api.py")

---

## ⚠️ Common Pitfalls

### 1. Blocking Operations
```python
# ❌ BAD - Blocks the async event loop
@app.get("/slow")
async def slow_endpoint():
    time.sleep(5)  # Blocks everything!
    return {"done": True}

# ✅ GOOD - Use async operations
@app.get("/slow")
async def slow_endpoint():
    await asyncio.sleep(5)  # Doesn't block
    return {"done": True}
```

### 2. No Error Handling
```python
# ❌ BAD - Unhandled errors crash the server
@app.post("/api")
async def endpoint(data: dict):
    return data["required_key"]  # KeyError!

# ✅ GOOD - Handle errors gracefully
@app.post("/api")
async def endpoint(data: dict):
    try:
        return data["required_key"]
    except KeyError:
        raise HTTPException(400, "Missing required_key")
```

### 3. Missing Validation
```python
# ❌ BAD - No validation
@app.post("/api")
async def endpoint(prompt: str):
    return {"result": prompt}

# ✅ GOOD - Pydantic validation
class Request(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=1000)

@app.post("/api")
async def endpoint(request: Request):
    return {"result": request.prompt}
```

### 4. No Rate Limiting
```python
# In production, add rate limiting:
from slowapi import Limiter
from slowapi.util import get_remote_address

limiter = Limiter(key_func=get_remote_address)

@app.post("/api")
@limiter.limit("5/minute")
async def endpoint():
    return {"result": "..."}
```

---

## 🏭 Production Checklist

Before deploying to production:

### Security
- [ ] API key authentication
- [ ] Rate limiting
- [ ] Input validation (Pydantic)
- [ ] CORS configuration
- [ ] HTTPS only

### Reliability
- [ ] Error handling
- [ ] Logging (structured)
- [ ] Health checks
- [ ] Timeouts
- [ ] Retry logic

### Performance
- [ ] Async operations
- [ ] Connection pooling
- [ ] Caching (Redis)
- [ ] Load balancing
- [ ] Monitoring (latency, errors)

### Documentation
- [ ] API docs (auto-generated)
- [ ] Example requests
- [ ] Error codes
- [ ] Versioning

### Deployment
- [ ] Dockerized
- [ ] Environment variables
- [ ] CI/CD pipeline
- [ ] Monitoring/alerting

---

## ✅ Summary

### What You Built

1. ✅ **FastAPI Application**
   - Multiple endpoints
   - Request/response validation
   - Error handling
   - Auto documentation

2. ✅ **LLM Integration**
   - Completion endpoint
   - Summarization endpoint
   - Classification endpoint

3. ✅ **Production Features**
   - Logging
   - Performance metrics
   - CORS middleware
   - Health checks

### Key Takeaways

⚡ **FastAPI is perfect for LLMs** - Async, fast, type-safe

📋 **Pydantic validates automatically** - No manual checks

📚 **Docs are automatic** - Interactive Swagger UI

🛡️ **Error handling is critical** - Never expose raw errors

📊 **Monitor everything** - Latency, errors, usage

### Next Steps

Continue building production APIs:

1. **Streaming:** `08_production_apis/03_streaming_api.ipynb`
2. **Authentication:** `08_production_apis/07_authentication.ipynb`
3. **Deployment:** `13_docker_containers/04_dockerizing_fastapi.ipynb`

---

## 🎉 Congratulations!

You built a production-ready LLM API!

**This is the foundation for ALL production AI systems:**
- Chatbots → Use this API
- RAG systems → Expose via this API
- Content generation → Serve with this API

**You can now deploy your AI applications to the world!** 🚀

---